# FSD AI vs Real Image Detection (Kaggle)
This notebook runs the Forensic Self-Descriptions (FSD) detector on a Kaggle dataset or a single uploaded image.

## What you need to do:
1) Turn **Internet ON** in Kaggle Notebook settings
2) Select a **GPU** accelerator (T4 or P100)
3) Add the dataset in Kaggle (so it appears under `/kaggle/input/ai_image_dataset/`)
4) Run cells in order

In [ ]:
# Clone repo and install dependencies
!git clone https://github.com/ductai199x/Forensic-Self-Descriptions-CVPR25.git
!pip -q install uv
!cd Forensic-Self-Descriptions-CVPR25 && uv pip install --system -e .

## Evaluate on the Dataset
This cell reads images from `/kaggle/input/ai_image_dataset/benchmark-v0.1/images/` (ai vs real), scores them, and prints accuracy, precision, recall, and F1.

## Quick EDA (before evaluation)
Counts total images, class balance, and per-category counts under `images/ai` and `images/real`.

In [ ]:
"""
SKIPPED — optional EDA, not needed to serve the API and its `assert` would HALT
Run All if the dataset isn't attached this session. Re-enable only if you want
to inspect the dataset.

from pathlib import Path
from collections import Counter

mounted_root = Path("/kaggle/input/datasets/kmazd1110/ai-image-dataset/benchmark-v0.1/images")
downloaded_root = Path("/kaggle/working/benchmark-v0.1/images")
dataset_root = mounted_root if mounted_root.exists() else downloaded_root
assert dataset_root.exists(), "Dataset not found. Add the dataset in Kaggle or run the download cell."

exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
class_counts = Counter()
category_counts = Counter()

for class_dir in ["ai", "real"]:
    base = dataset_root / class_dir
    if not base.exists():
        continue
    for p in base.rglob("*"):
        if p.suffix.lower() in exts:
            class_counts[class_dir] += 1
            # category is the first folder under class (ai/animal, real/portrait, etc.)
            parts = p.relative_to(base).parts
            category = parts[0] if len(parts) > 1 else "(uncategorized)"
            category_counts[(class_dir, category)] += 1

total = sum(class_counts.values())
print(f"Dataset root: {dataset_root}")
print(f"Total images: {total}")
print(f"AI images: {class_counts.get('ai', 0)}")
print(f"Real images: {class_counts.get('real', 0)}")
print("\\nTop categories by class:")
for (cls, cat), count in category_counts.most_common(20):
    print(f"{cls}/{cat}: {count}")
"""


In [ ]:
"""
SKIPPED — this is the ~2.5 hour benchmark eval (720 images). Not needed to serve
the API: the FastAPI cell below loads its own FSDDetector. Its `assert` would also
HALT Run All if the dataset isn't attached this session. Re-enable only if you want
to re-run the benchmark.

import os
import sys
import random
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Add repo to path
repo_path = "/kaggle/working/Forensic-Self-Descriptions-CVPR25"
if repo_path not in sys.path:
    sys.path.append(repo_path)

from fsd import FSDDetector

mounted_root = Path("/kaggle/input/datasets/kmazd1110/ai-image-dataset/benchmark-v0.1/images")
downloaded_root = Path("/kaggle/input/datasets/kmazd1110/ai-image-dataset/benchmark-v0.1/images")
dataset_root = mounted_root if mounted_root.exists() else downloaded_root
assert dataset_root.exists(), "Dataset not found. Add the dataset in Kaggle or run the download cell."

samples_per_category = 60
random_seed = 42


def build_from_dirs(root: Path, samples_per_cat: int, seed: int):
    exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    rng = random.Random(seed)
    paths = []
    labels = []

    for label_dir, label in [("real", 0), ("ai", 1)]:
        base = root / label_dir
        if not base.exists():
            continue

        category_dirs = [p for p in base.iterdir() if p.is_dir()]
        if not category_dirs:
            category_dirs = [base]

        for cat_dir in category_dirs:
            candidates = [p for p in cat_dir.rglob("*") if p.suffix.lower() in exts]
            if not candidates:
                continue
            take = min(samples_per_cat, len(candidates))
            sampled = rng.sample(candidates, take)
            paths.extend([str(p) for p in sampled])
            labels.extend([label] * take)

    if not paths:
        raise ValueError("No labeled images found under images/ai and images/real.")
    return paths, labels


image_paths, y_true = build_from_dirs(dataset_root, samples_per_category, random_seed)

print(f"Scoring {len(image_paths)} images ({samples_per_category} per category, per class)...")
detector = FSDDetector.load(attribution=False)
results = detector.score_batch(image_paths)
y_pred = [1 if r.is_fake else 0 for r in results]

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print("\\n--- Overall Metrics ---")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")

print("\\n--- Per-Model Metrics ---")
model_names = []
for p, y in zip(image_paths, y_true):
    if y == 1:
        cat = Path(p).parent.name
        stem = Path(p).stem
        if f"_{cat}_" in stem:
            model_names.append(stem.split(f"_{cat}_")[0])
        else:
            model_names.append("unknown_model")
    else:
        model_names.append("real")

ai_models = sorted(list(set(m for m, y in zip(model_names, y_true) if y == 1)))

for model in ai_models:
    indices = [i for i, m in enumerate(model_names) if m == model or m == "real"]
    y_t = [y_true[i] for i in indices]
    y_p = [y_pred[i] for i in indices]

    if len(set(y_t)) > 1:
        m_acc = accuracy_score(y_t, y_p)
        m_prec = precision_score(y_t, y_p, zero_division=0)
        m_rec = recall_score(y_t, y_p, zero_division=0)
        m_f1 = f1_score(y_t, y_p, zero_division=0)

        print(f"\\nModel: {model}")
        print(f"  Accuracy:  {m_acc:.4f}")
        print(f"  Precision: {m_prec:.4f}")
        print(f"  Recall:    {m_rec:.4f}")
        print(f"  F1:        {m_f1:.4f}")
"""


## Serve FSD as an API for the Verilens extension

The two cells below turn this notebook into the **AI Image Detection** backend the
Verilens Chrome extension talks to:

1. **Serve** — load `FSDDetector` once and expose a tiny FastAPI app:
   - `GET /health` → `{ ok, model }`
   - `POST /detect` with `{ image_b64 }` → `{ is_fake, score }`
2. **Tunnel** — open an ngrok HTTPS tunnel so your local browser can reach it.
   Paste the printed URL into `lib/config.local.js` as `imageBackendUrl`.

> Note: FSD scores ~12s/image on a Kaggle T4, so a single "Check media" click
> takes a few seconds — the extension shows a loading state meanwhile.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Serve FSD over HTTP (FastAPI + uvicorn in a background thread)
# ─────────────────────────────────────────────────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "-q", "fastapi", "uvicorn[standard]", "nest_asyncio", "pillow"], check=True)

import sys, base64, tempfile, os, threading
import nest_asyncio, uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

# Make the cloned repo importable (same path the eval cell used).
repo_path = "/kaggle/working/Forensic-Self-Descriptions-CVPR25"
if repo_path not in sys.path:
    sys.path.append(repo_path)
from fsd import FSDDetector

# Load the detector ONCE at startup (this is the slow part — keep it warm).
print("Loading FSDDetector...")
DETECTOR = FSDDetector.load(attribution=False)
print("FSDDetector ready.")


def _extract_score(res):
    """FSD's result exposes .is_fake; some builds also carry a numeric score.
    Pull the first float-ish attribute we recognise, else None (the extension
    falls back to a default probability when score is null)."""
    for attr in ("score", "prob", "probability", "p_fake", "fake_score", "pvalue", "p_value"):
        v = getattr(res, attr, None)
        if isinstance(v, (int, float)):
            return float(v)
    return None


def score_one(image_path: str):
    res = DETECTOR.score_batch([image_path])[0]
    return bool(res.is_fake), _extract_score(res)


app = FastAPI()
# The extension calls cross-origin from the social site's page context.
app.add_middleware(
    CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"]
)


class DetectIn(BaseModel):
    image_b64: str


@app.get("/health")
def health():
    return {"ok": True, "model": "FSD"}


@app.post("/detect")
def detect(body: DetectIn):
    b64 = body.image_b64 or ""
    if "," in b64[:64]:          # tolerate a data:URL prefix
        b64 = b64.split(",", 1)[1]
    raw = base64.b64decode(b64)
    tmp = tempfile.NamedTemporaryFile(suffix=".jpg", delete=False)
    try:
        tmp.write(raw)
        tmp.flush()
        tmp.close()
        is_fake, score = score_one(tmp.name)
        return {"is_fake": is_fake, "score": score}
    finally:
        try:
            os.unlink(tmp.name)
        except OSError:
            pass


# Run uvicorn in a daemon thread so this cell returns and the next cell (ngrok)
# can open the tunnel while the server keeps serving.
nest_asyncio.apply()
_config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
_server = uvicorn.Server(_config)
threading.Thread(target=_server.run, daemon=True).start()
print("FSD server listening on http://0.0.0.0:8000  (/health, /detect)")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Expose the FSD server to the internet with ngrok
# ─────────────────────────────────────────────────────────────────────────────
# The FSD server above only listens on localhost:8000, which the Verilens Chrome
# extension (running on your own machine) can't reach. ngrok opens a public HTTPS
# tunnel to it. Copy the printed URL into:
#     lib/config.local.js  ->  imageBackendUrl
#
# IMPORTANT if you also run videoveritas-ai-video-detection.ipynb and/or
# fast-gpt.ipynb at the same time: ngrok's free plan assigns each ACCOUNT one
# shared static domain. If multiple notebooks use the same authtoken, the later
# tunnels fail with "ERR_NGROK_334: endpoint ... is already online". Fix: use a
# SEPARATE free ngrok account (and authtoken) for each notebook.
#   - This notebook reads the Kaggle Secret NGROK_AUTH_TOKEN_IMAGE (falls back
#     to NGROK_AUTH_TOKEN if that's the only one you've set up).
#   - videoveritas-ai-video-detection.ipynb reads NGROK_AUTH_TOKEN_VIDEO.
#   - fast-gpt.ipynb reads NGROK_AUTH_TOKEN_TEXT.
#
# Get a free authtoken at: https://dashboard.ngrok.com/get-started/your-authtoken
# (Kaggle: add it as a Secret, or paste it inline below.)

!pip install -q pyngrok

from pyngrok import ngrok

def _try_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return ""

# Prefer a Kaggle Secret; fall back to the inline value.
NGROK_AUTH_TOKEN = ""  # ← paste your token here if not using Kaggle Secrets
NGROK_AUTH_TOKEN = (
    NGROK_AUTH_TOKEN
    or _try_secret("NGROK_AUTH_TOKEN_IMAGE")
    or _try_secret("NGROK_AUTH_TOKEN")
)

if not NGROK_AUTH_TOKEN:
    raise SystemExit("Set NGROK_AUTH_TOKEN_IMAGE (inline above or as a Kaggle Secret) and re-run this cell.")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()  # drop any tunnel from a previous run
public_url = ngrok.connect(8000, "http").public_url

print("=" * 70)
print("FSD image detector is now public at:")
print("   ", public_url)
print("=" * 70)
print("\nPaste that URL into the Verilens extension:")
print("   lib/config.local.js -> imageBackendUrl")
print("\nSanity check (should return {\"ok\": true, \"model\": \"FSD\"}):")
print(f"   curl {public_url}/health -H 'ngrok-skip-browser-warning: true'")
